In [8]:
import pandas as pd
import numpy as np

In [9]:
# 1. Load and sort chronologically (essential for time-based features later!)
df = pd.read_csv('data/atp_tennis.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

In [10]:
# 2. Filter for valid rankings AND valid ATP points (removes early-2000s -1 placeholders)
df = df[
    (df['Rank_1'] > 0) & (df['Rank_2'] > 0) &
    (df['Pts_1'] > 0) & (df['Pts_2'] > 0)
].dropna(subset=['Rank_1', 'Rank_2', 'Pts_1', 'Pts_2', 'Winner', 'Surface']).reset_index(drop=True)


In [11]:
# 3. Target variable (1 if Player 1 wins, 0 if Player 2 wins)
df['Player_1_Win'] = (df['Winner'] == df['Player_1']).astype(int)


In [12]:
# 4. Ranking & Points features (positive = Player 1 advantage)
df['Rank_Diff'] = df['Rank_2'] - df['Rank_1']
df['Log_Rank_Diff'] = np.log(df['Rank_2']) - np.log(df['Rank_1'])
df['Pts_Diff'] = df['Pts_1'] - df['Pts_2']

In [13]:
# 5. Re-check the "dumb" baseline on this exact filtered dataset
baseline_acc = ((df['Rank_1'] < df['Rank_2']).astype(int) == df['Player_1_Win']).mean()


In [14]:
print(f"Matches with valid rankings & points: {len(df)}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Dumb Baseline Accuracy on this subset: {baseline_acc:.3f}")

Matches with valid rankings & points: 53447
Date range: 2005-07-04 to 2026-11-01
Dumb Baseline Accuracy on this subset: 0.656


In [15]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

In [16]:
# 1. Define features (X) and target (y)
num_cols = ['Rank_Diff', 'Log_Rank_Diff', 'Pts_Diff', 'Best of']
cat_cols = ['Surface', 'Court']

X = df[num_cols + cat_cols]
y = df['Player_1_Win']


In [17]:
# 2. Train/Test Split (keep X_test locked away until the very end!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [18]:
# 3. Preprocessing Pipeline (scales numeric cols, one-hot encodes text cols)
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols)
])



In [19]:
# 4. Compare classifiers from class using 5-fold Cross-Validation on X_train
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Gaussian Naive Bayes': GaussianNB(),
    'KNN (k=25)': KNeighborsClassifier(n_neighbors=25),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
}



In [20]:
print(f"Dumb Baseline to beat: {baseline_acc:.3f}\n")
for name, clf in models.items():
    pipe = Pipeline([
        ('prep', preprocessor),
        ('clf', clf)
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
    print(f"{name:22s} | 5-Fold CV Accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

Dumb Baseline to beat: 0.656

Logistic Regression    | 5-Fold CV Accuracy: 0.656 (+/- 0.005)
Gaussian Naive Bayes   | 5-Fold CV Accuracy: 0.656 (+/- 0.005)
KNN (k=25)             | 5-Fold CV Accuracy: 0.638 (+/- 0.004)
Random Forest          | 5-Fold CV Accuracy: 0.656 (+/- 0.005)


In [21]:
# 1. Reshape matches into a chronological player-by-player log (2 rows per match)
p1_log = df[['Date', 'Surface', 'Player_1', 'Player_1_Win']].rename(
    columns={'Player_1': 'Player', 'Player_1_Win': 'Won'}
)
p1_log['Match_ID'] = df.index
p1_log['Slot'] = 1

p2_log = df[['Date', 'Surface', 'Player_2', 'Player_1_Win']].rename(
    columns={'Player_2': 'Player'}
)
p2_log['Won'] = 1 - p2_log['Player_1_Win']
p2_log['Match_ID'] = df.index
p2_log['Slot'] = 2

player_df = pd.concat([p1_log, p2_log], ignore_index=True).sort_values(['Date', 'Match_ID']).reset_index(drop=True)



In [22]:
# 2. Leak-free rolling features (shift(1) ensures we ONLY use past matches!)
# Overall recent form (last 10 matches, defaulted to 0.5 for brand-new players)
player_df['Form_10'] = (
    player_df.groupby('Player')['Won']
    .transform(lambda s: s.shift(1).rolling(10, min_periods=1).mean())
    .fillna(0.5)
)

# Surface-specific historical win rate (expanding average of all prior matches on that surface)
player_df['Surface_WinRate'] = (
    player_df.groupby(['Player', 'Surface'])['Won']
    .transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
    .fillna(0.5)
)

# Career total matches played prior to today (experience)
player_df['Matches_Played'] = player_df.groupby('Player').cumcount()



In [23]:
# 3. Merge these leak-free stats back onto the main match dataframe
p1_feats = player_df[player_df['Slot'] == 1].set_index('Match_ID')[['Form_10', 'Surface_WinRate', 'Matches_Played']]
p2_feats = player_df[player_df['Slot'] == 2].set_index('Match_ID')[['Form_10', 'Surface_WinRate', 'Matches_Played']]

df['Form10_Diff'] = p1_feats['Form_10'] - p2_feats['Form_10']
df['Surface_WinRate_Diff'] = p1_feats['Surface_WinRate'] - p2_feats['Surface_WinRate']
df['Exp_Diff'] = p1_feats['Matches_Played'] - p2_feats['Matches_Played']

print("New player-history features created successfully!")
print(df[['Player_1', 'Player_2', 'Surface', 'Form10_Diff', 'Surface_WinRate_Diff', 'Exp_Diff']].tail(5))

New player-history features created successfully!
           Player_1    Player_2 Surface  Form10_Diff  Surface_WinRate_Diff  \
53442   Struff J.L.  Lehecka J.    Hard          0.1             -0.156956   
53443   Vacherot V.    Fritz T.    Hard          0.1              0.001633   
53444   Medvedev D.  Alcaraz C.    Hard          0.1             -0.039008   
53445   Vacherot V.  Lehecka J.    Hard          0.0              0.022644   
53446  De Minaur A.  Alcaraz C.    Hard          0.0             -0.132685   

       Exp_Diff  
53442       269  
53443      -492  
53444       258  
53445      -166  
53446       120  


In [24]:
# Track prior head-to-head wins between each unique pair of players (chronological, leak-free)
h2h_wins = {}
h2h_diff_list = []

for row in df.itertuples():
    p1, p2, winner = row.Player_1, row.Player_2, row.Winner
    
    # Look up prior wins against each other BEFORE today's match
    p1_prior = h2h_wins.get((p1, p2), 0)
    p2_prior = h2h_wins.get((p2, p1), 0)
    h2h_diff_list.append(p1_prior - p2_prior)
    
    # Update the dictionary AFTER recording pre-match state
    if winner == p1:
        h2h_wins[(p1, p2)] = p1_prior + 1
    else:
        h2h_wins[(p2, p1)] = p2_prior + 1

df['H2H_Diff'] = h2h_diff_list

print("H2H_Diff created! Sample matches with prior head-to-head history:")
print(df[df['H2H_Diff'] != 0][['Date', 'Player_1', 'Player_2', 'H2H_Diff', 'Player_1_Win']].tail(5))

H2H_Diff created! Sample matches with prior head-to-head history:
            Date      Player_1    Player_2  H2H_Diff  Player_1_Win
53441 2026-10-31  De Minaur A.   Zverev A.        -8             1
53442 2026-10-31   Struff J.L.  Lehecka J.        -3             0
53443 2026-10-31   Vacherot V.    Fritz T.        -1             1
53444 2026-10-31   Medvedev D.  Alcaraz C.        -3             0
53446 2026-11-01  De Minaur A.  Alcaraz C.        -6             0


In [25]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# 1. Full feature set (rankings, points, form, surface history, experience, H2H, court context)
num_cols = [
    'Rank_Diff', 'Log_Rank_Diff', 'Pts_Diff', 'Best of',
    'Form10_Diff', 'Surface_WinRate_Diff', 'Exp_Diff', 'H2H_Diff'
]
cat_cols = ['Surface', 'Court']

X = df[num_cols + cat_cols]
y = df['Player_1_Win']

# 2. Train/Test Split (keep X_test locked away until final evaluation)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Preprocessing Pipeline (prevents data leakage during CV)
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols)
])

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Gaussian Naive Bayes': GaussianNB(),
    'KNN (k=35)': KNeighborsClassifier(n_neighbors=35),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42, n_jobs=-1)
}

print(f"Dumb Baseline to beat: {baseline_acc:.3f}\n")
for name, clf in models.items():
    pipe = Pipeline([
        ('prep', preprocessor),
        ('clf', clf)
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
    print(f"{name:22s} | 5-Fold CV Accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

Dumb Baseline to beat: 0.656

Logistic Regression    | 5-Fold CV Accuracy: 0.665 (+/- 0.003)
Gaussian Naive Bayes   | 5-Fold CV Accuracy: 0.663 (+/- 0.002)
KNN (k=35)             | 5-Fold CV Accuracy: 0.654 (+/- 0.004)
Random Forest          | 5-Fold CV Accuracy: 0.666 (+/- 0.004)


In [26]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# 1. Build Pipeline for our top CV model (Random Forest)
rf_pipe = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
])

# 2. Tune hyperparameters using 5-fold CV on X_train ONLY
param_grid = {
    'clf__max_depth': [6, 10, 14],
    'clf__min_samples_leaf': [5, 20]
}

grid = GridSearchCV(rf_pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

print(f"Best CV Accuracy: {grid.best_score_:.3f}")
print(f"Best Parameters:  {grid.best_params_}\n")

# 3. Final evaluation on the held-out Test Set (X_test)
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print(f"Final Test Accuracy: {accuracy_score(y_test, y_pred):.3f} (vs. {baseline_acc:.3f} baseline)")
print(f"Final Test ROC-AUC:  {roc_auc_score(y_test, y_prob):.3f}\n")
print("Classification Report:\n", classification_report(y_test, y_pred))

# 4. Extract top feature importances to see what drove predictions
feature_names = best_model.named_steps['prep'].get_feature_names_out()
importances = pd.Series(
    best_model.named_steps['clf'].feature_importances_,
    index=feature_names
).sort_values(ascending=False)

print("Top 10 Most Important Features:")
print(importances.head(10).round(4))

Best CV Accuracy: 0.667
Best Parameters:  {'clf__max_depth': 10, 'clf__min_samples_leaf': 5}

Final Test Accuracy: 0.672 (vs. 0.656 baseline)
Final Test ROC-AUC:  0.737

Classification Report:
               precision    recall  f1-score   support

           0       0.67      0.67      0.67      5345
           1       0.67      0.67      0.67      5345

    accuracy                           0.67     10690
   macro avg       0.67      0.67      0.67     10690
weighted avg       0.67      0.67      0.67     10690

Top 10 Most Important Features:
num__Log_Rank_Diff           0.2904
num__Pts_Diff                0.2102
num__Surface_WinRate_Diff    0.1517
num__Rank_Diff               0.1444
num__Exp_Diff                0.0852
num__Form10_Diff             0.0728
num__H2H_Diff                0.0205
num__Best of                 0.0090
cat__Surface_Hard            0.0044
cat__Surface_Clay            0.0042
dtype: float64
